# Section 2a: Structural Variant — Copy Number Variation (CNV) Analysis

Welcome to **Section 2a: Structural Variant - Copy Number Variation (CNV) Analysis**. In this notebook, we cover the theoretical foundations and hands-on pipeline execution of tumor-only copy number calling using **`CNVkit`**.

### Workflow Steps:
1. **Target BED Bin Creation** (`cnvkit.py target`)
2. **Antitarget BED Bin Creation** (`cnvkit.py access` & `cnvkit.py antitarget`)
3. **Coverage Computation** (`cnvkit.py coverage` for targets & antitargets)
4. **Flat Reference Construction** (`cnvkit.py reference`)
5. **Fix & Log2 Coverage Normalization** (`cnvkit.py fix`)
6. **Genomic Segmentation** (`cnvkit.py segment`)
7. **Purity, Ploidy & Absolute Copy Number Calling** (`cnvkit.py call`)
8. **B-Allele Frequency (BAF) & LOH Integration** (`cnvkit.py call --vcf`)
9. **Pandas Data Display & CNV Segment Inspection**


---  
## Step 1: Target Region Bin Preparation (`cnvkit.py target`)

### Theoretical Background

#### Target Bins in Exome/Panel Sequencing
In targeted capture sequencing (WES or gene panels), hybrid capture baits target specific exonic genomic regions defined in a BED file. `cnvkit.py target` prepares these intervals for depth counting:

- **Target Splitting**: Extremely large exons or target regions are subdivided into smaller, roughly equal-sized bins (~200-500 bp) to maintain uniform variance in read depth counts.
- **Target Trimming**: Small target gaps are merged or trimmed to reduce noise at target boundaries.


In [ ]:
%%bash
mkdir -p Results/CNV
cnvkit.py target References/Target_regions_chr17.bed --split -o Results/CNV/chr17_target.bed


---  
## Step 2: Access & Antitarget Bin Derivation (`cnvkit.py access` & `cnvkit.py antitarget`)

### Theoretical Background

#### Off-Target Read Capture ("Antitargets")
During hybrid capture library prep, non-specific binding causes thousands of reads to sequence off-target intronic and intergenic regions. Standard CNV tools discard off-target reads as noise; however, `CNVkit` utilizes off-target coverage as large, low-resolution background bins:

1. **`cnvkit.py access`**: Identifies accessible (non-N-masked) regions of the reference genome FASTA (`chr17.fa`).
2. **`cnvkit.py antitarget`**: Derives off-target background bins by taking the accessible genome minus the target capture BED regions.

> 💡 **Bioinformatics Interview Q&A: Target vs Antitarget Bins**
>
> **Q: Why does CNVkit utilize off-target ("antitarget") bins in target enrichment datasets, and how do target and antitarget bin sizes differ?**
>
> **A:** Off-target reads originate from non-specific hybrid capture binding in intronic and intergenic regions. While individual off-target read density is low, grouping them into large antitarget bins (~100–500 kb) provides a smooth, low-resolution background coverage profile across the entire genome. Target bins are small and high-resolution (~200-500 bp). Combining both allows CNVkit to detect localized single-exon deletions alongside broad chromosome-arm copy number alterations.


In [ ]:
%%bash
cnvkit.py access References/chr17.fa -o Results/CNV/access_chr17.bed
cnvkit.py antitarget Results/CNV/chr17_target.bed -g Results/CNV/access_chr17.bed -o Results/CNV/chr17_antitarget.bed


---  
## Step 3: Bin Read Depth Coverage Computation (`cnvkit.py coverage`)

### Theoretical Background

#### Normalizing Raw Read Counts
`cnvkit.py coverage` calculates the raw read depth in each target and antitarget bin from the deduplicated alignment file (`dedup.bam`). Output files (`.targetcoverage.cnn` and `.antitargetcoverage.cnn`) contain:
- Read counts per bin
- Mean log2 coverage depth ($depth_{raw}$)


In [ ]:
%%bash
cnvkit.py coverage Results/Deduplication/dedup.bam Results/CNV/chr17_target.bed -o Results/CNV/TumorSample.targetcoverage.cnn
cnvkit.py coverage Results/Deduplication/dedup.bam Results/CNV/chr17_antitarget.bed -o Results/CNV/TumorSample.antitargetcoverage.cnn


---  
## Step 4: Flat Reference Construction (`cnvkit.py reference`)

### Theoretical Background

#### Tumor-Only Baseline Normalization
When a matched normal sample is unavailable, `cnvkit.py reference` constructs a **flat reference baseline** (`FlatReference.cnn`) using sequence properties from the reference genome FASTA (`chr17.fa`):

- **GC Content Bias Model**: Bins with extreme GC content ($< 30\%$ or $> 65\%$) systematically exhibit reduced sequencing coverage.
- **RepeatMasker Bias Model**: Repetitive elements reduce unique alignment confidence.

> 💡 **Bioinformatics Interview Q&A: CNV Reference Baselines**
>
> **Q: What is a CNV reference baseline and how does a flat reference differ from a Panel of Normals (PoN) baseline?**
>
> **A:** A CNV baseline models technical background noise across genomic intervals. A Panel of Normals (PoN) baseline uses dozens of healthy samples processed with the exact same capture kit to model probe-specific capture efficiencies and batch effects. A **flat reference** assumes neutral 1.0 depth across all bins and applies mathematical GC-content and RepeatMasker regressions to normalize coverage in tumor-only settings where pooled normal samples are absent.


In [ ]:
%%bash
cnvkit.py reference -f References/chr17.fa -o Results/CNV/FlatReference.cnn --target Results/CNV/chr17_target.bed --antitarget Results/CNV/chr17_antitarget.bed


---  
## Step 5: Coverage Fix & Log2 Ratio Normalization (`cnvkit.py fix`)

### Theoretical Background

#### Log2 Copy Ratio Calculation
`cnvkit.py fix` combines target and antitarget coverages, subtracts expected log2 coverage from the reference baseline, and applies bi-weight location regression to output normalized log2 copy ratios (`.cnr` file):

$$\text{log2\_ratio} = \log_2 \left( \frac{\text{Tumor Coverage}}{\text{Reference Baseline Coverage}} \right)$$

- **$\text{log2} = 0.0$**: Neutral diploid copy number ($CN = 2$).
- **$\text{log2} > +0.2$**: Heterozygous/Homozygous Gain or Amplification ($CN \ge 3$).
- **$\text{log2} < -0.25$**: Heterozygous or Homozygous Loss/Deletion ($CN \le 1$).


In [ ]:
%%bash
cnvkit.py fix Results/CNV/TumorSample.targetcoverage.cnn Results/CNV/TumorSample.antitargetcoverage.cnn Results/CNV/FlatReference.cnn -o Results/CNV/TumorSample.cnr


---  
## Step 6: Genome Segmentation (`cnvkit.py segment`)

### Theoretical Background

#### Segmenting Noisy Bin Ratios
Individual genomic bins exhibit high local noise. Segmentation algorithms scan adjacent bins across chromosomes to group regions sharing statistically identical log2 ratios into discrete contiguous segments (`.cns` file):

- **Haar Wavelet / Circular Binary Segmentation (CBS)**: Identifies changepoints/breakpoints where copy number status shifts.

> 💡 **Bioinformatics Interview Q&A: Segmentation Algorithms**
>
> **Q: How do Circular Binary Segmentation (CBS) and Haar wavelet algorithms identify CNV breakpoint changepoints?**
>
> **A:** Segmentation algorithms scan sequential bin log2 ratios to test the null hypothesis that adjacent bins share the same mean log2 ratio. CBS uses recursive t-test statistics to find changepoints that maximize differences in mean log2 ratios between adjacent segments. Haar wavelet segmentation uses multi-resolution wavelet transforms to smooth bin-level noise while preserving sharp step-function copy number boundaries.


In [ ]:
%%bash
cnvkit.py segment -m haar Results/CNV/TumorSample.cnr -o Results/CNV/TumorSample.cns


---  
## Step 7 & 8: Purity, Ploidy & B-Allele Frequency (BAF) Calling (`cnvkit.py call`)

### Theoretical Background

#### 1. Integer Copy Number Calling
Log2 ratios represent continuous signals. `cnvkit.py call` translates continuous log2 segment ratios into discrete integer copy number states ($CN = 0, 1, 2, 3, 4+$).

#### 2. B-Allele Frequency (BAF) & Loss of Heterozygosity (LOH)
Log2 coverage ratios alone cannot distinguish between copy-neutral Loss of Heterozygosity (cnLOH) and normal diploid regions. By ingesting the GATK Mutect2 somatic/germline VCF (`filtered_somatic_variants.vcf.gz`), `CNVkit` extracts **B-Allele Frequencies (BAF)** at heterozygous SNP loci:

- **Diploid Heterozygous SNP**: $BAF \approx 0.5$ (50% Ref, 50% Alt).
- **Allelic Imbalance / LOH**: $BAF$ shifts toward $0.0$ or $1.0$, signaling loss of one parental allele.

> 💡 **Bioinformatics Interview Q&A: BAF & Loss of Heterozygosity (LOH)**
>
> **Q: What is B-Allele Frequency (BAF), and why is BAF critical for resolving Copy-Neutral Loss of Heterozygosity (cnLOH)?**
>
> **A:** B-Allele Frequency ($BAF = B / (A + B)$) measures the ratio of alternate allele reads at heterozygous SNP sites. In a normal diploid genome, heterozygous BAF clusters around $0.5$. In physical deletions, read depth drops ($\log_2 < -0.25$) and BAF shifts to $0.0$ or $1.0$. In **Copy-Neutral LOH (cnLOH)** (uniparental disomy), one allele is lost while the other is duplicated: read depth remains unchanged ($\log_2 \approx 0.0$), but BAF splits to $0.0$ and $1.0$. Combining depth ratios with BAF is the only way to detect cnLOH.


In [ ]:
%%bash
cnvkit.py call Results/CNV/TumorSample.cns \
  --vcf Results/Variants/filtered_somatic_variants.vcf.gz \
  -o Results/CNV/TumorSample.call.cns


### Inspect CNV Calls & Copy Number Alteration Table (Pandas)

We load the generated `.cnr`, `.cns`, and `.call.cns` files using Pandas to inspect copy number gains, losses, and BAF calls.

In [ ]:
import pandas as pd
import os

# 1. Load Copy Ratio Bins (.cnr)
cnr_file = "Results/CNV/TumorSample.cnr"
if os.path.exists(cnr_file):
    df_cnr = pd.read_csv(cnr_file, sep="\t")
    print("=== Normalized Log2 Ratio Bins (.cnr - First 15 Bins) ===")
    display(df_cnr.head(15))

# 2. Load Segmented Copy Number Calls (.call.cns)
call_file = "Results/CNV/TumorSample.call.cns"
if os.path.exists(call_file):
    df_call = pd.read_csv(call_file, sep="\t")
    print("\n=== Segmented Copy Number Calls (.call.cns - All Segments) ===")
    display(df_call)
    
    # Filter for Copy Number Alterations (CN != 2)
    cna_alterations = df_call[df_call["cn"] != 2]
    print(f"\n=== Identified Copy Number Alteration Segments (CN != 2): {len(cna_alterations)} ===")
    if not cna_alterations.empty:
        display(cna_alterations)
    else:
        print("No significant broad copy number gains or losses detected on chr17 (Diploid baseline maintained).")
